# LexiLensAI SDK — Production Demo

This notebook demonstrates the **production LexiLensAI SDK** (v0.1.0, published on PyPI) instrumenting a real multi-agent workflow.

**What you'll see:**
- One-line SDK initialization
- Automatic instrumentation of Strands agents
- Session-aware telemetry with parent-child tracking
- JSONL exporter (no infrastructure required)
- Session summary with delegation graph

**No prototype code** — this uses the `lexilensai-sdk` package from PyPI.

---

## Installation

Install the SDK and Strands framework:

In [ ]:
!pip install -q lexilensai-sdk strands-agents boto3

## Step 1: Initialize Session Instrumentation (One Line)

This is all you need. The SDK automatically patches Strands agents and emits session-aware spans.

In [ ]:
from lexilensai import LexiLens

# Initialize with JSONL exporter (writes to local file — no backend needed)
lexilens = LexiLens.init(
    tenant_id="demo",
    application_id="travel_planner",
    exporter="jsonl",  # Writes to lexilens_spans.jsonl
    objective="Plan a multi-city European trip"
)

print(f"✓ LexiLens initialized")
print(f"  Session ID: {lexilens.session_id}")
print(f"  Spans file: lexilens_spans.jsonl")

## Step 2: Define Specialist Agents

Standard Strands agents — **no instrumentation code needed**. The SDK automatically captures all agent calls.

In [ ]:
from strands import Agent, tool
import os

# Bypass Strands tool consent prompts for demo
os.environ["BYPASS_TOOL_CONSENT"] = "true"

@tool
def research_destination(city: str) -> str:
    """Research tourist information about a destination."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="""You are a travel research specialist. 
        Provide concise, factual information about destinations including:
        - Top attractions
        - Best time to visit
        - Average costs
        Keep responses under 200 words."""
    )
    return str(agent(f"Research {city} as a tourist destination"))

@tool
def create_itinerary(destination: str, days: int) -> str:
    """Create a day-by-day travel itinerary."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="""You are a trip planning specialist.
        Create practical, day-by-day itineraries with:
        - Morning, afternoon, evening activities
        - Travel times between locations
        - Meal recommendations
        Be concise — bullet points preferred."""
    )
    return str(agent(f"Create a {days}-day itinerary for {destination}"))

@tool
def budget_estimate(destination: str, days: int) -> str:
    """Estimate travel costs for a destination."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="""You are a travel budget specialist.
        Provide realistic cost estimates including:
        - Accommodation (budget/mid/luxury)
        - Food (daily average)
        - Attractions (total)
        - Local transport
        Return total per-person estimates."""
    )
    return str(agent(f"Estimate costs for {days} days in {destination}"))

print("✓ Specialist agents defined")

## Step 3: Define Orchestrator Agent

The orchestrator coordinates the specialist agents. Watch how the SDK tracks delegation automatically.

In [ ]:
orchestrator = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are a travel planning coordinator.
    
    Use your tools to help users plan trips:
    - research_destination: Get destination info
    - create_itinerary: Plan day-by-day activities
    - budget_estimate: Calculate costs
    
    For comprehensive trip planning, use multiple tools in sequence.
    Synthesize the information into a cohesive response.""",
    tools=[research_destination, create_itinerary, budget_estimate]
)

print("✓ Orchestrator created with 3 specialist tools")

## Step 4: Run the Multi-Agent Workflow

Execute a query that will trigger the orchestrator to delegate to multiple specialists.

In [ ]:
query = "Plan a 4-day trip to Barcelona with budget breakdown"

print(f"Query: {query}\n")
print("=" * 60)

response = orchestrator(query)

print("=" * 60)
print(f"\n✓ Workflow complete")
print(f"  The SDK captured all agent calls, delegations, and metadata.")

## Step 5: Close Session and Examine Telemetry

Close the session (emits `session.end` span) and read back the JSONL file to see what was captured.

In [ ]:
# Close session
lexilens.close()
print("✓ Session closed\n")

# Read back spans
import json

with open("lexilens_spans.jsonl", "r") as f:
    spans = [json.loads(line) for line in f]

print(f"Captured {len(spans)} spans:\n")

# Display span summary
for i, span in enumerate(spans, 1):
    span_name = span.get("span_name", "unknown")
    agent_id = span.get("attributes", {}).get("agent_id", "unknown")
    parent_id = span.get("attributes", {}).get("parent_span_id")
    
    indent = "  " if parent_id else ""
    print(f"{i:2}. {indent}{span_name:20} agent={agent_id}")

print("\n" + "=" * 60)

## Step 6: Visualize Session Structure

Build a simple parent-child tree from the captured spans.

In [ ]:
# Build agent call tree
agent_calls = [s for s in spans if s.get("span_name") in ["agent.start", "agent.end"]]
agents_seen = set()
delegations = []

for span in agent_calls:
    attrs = span.get("attributes", {})
    agent_id = attrs.get("agent_id")
    parent_span_id = attrs.get("parent_span_id")
    
    if agent_id:
        agents_seen.add(agent_id)
    
    # Find parent agent
    if parent_span_id and span.get("span_name") == "agent.start":
        for parent_span in agent_calls:
            if parent_span.get("attributes", {}).get("span_id") == parent_span_id:
                parent_agent = parent_span.get("attributes", {}).get("agent_id")
                if parent_agent and parent_agent != agent_id:
                    delegations.append((parent_agent, agent_id))
                break

print("\nSession Summary:")
print(f"  Agents invoked: {len(agents_seen)}")
print(f"  Agents: {', '.join(sorted(agents_seen))}")

if delegations:
    print(f"\n  Delegation Graph:")
    for parent, child in set(delegations):
        print(f"    {parent} → {child}")

print("\n" + "=" * 60)

## Next Steps: Production Integration

This demo used the **JSONL exporter** (writes to local file). For production:

### 1. Switch to OTel Exporter

Send spans to an OpenTelemetry collector:

```python
lexilens = LexiLens.init(
    tenant_id="your_company",
    application_id="production_app",
    exporter="otel",
    collector_endpoint="http://localhost:4317"  # Your OTel collector
)
```

### 2. Connect to LexiLensAI Platform

Point your OTel collector to the LexiLensAI ingestion endpoint:

```yaml
# otel-collector-config.yaml
exporters:
  otlphttp:
    endpoint: https://api.lexilensai.com/v1/traces
    headers:
      x-api-key: ${LEXILENS_API_KEY}
```

### 3. Platform Features

Once connected, the platform provides:
- Session reconstruction with execution graphs
- Anomaly detection (retry storms, token explosions)
- Root cause analysis with LLM-generated explanations
- Cost attribution per session and per agent
- Timeline replay and debugging
- Governance policy evaluation

See: https://github.com/curvsort/lexilensai

---

## Resources

- **SDK GitHub:** https://github.com/curvsort/lexilensai-sdk
- **PyPI Package:** https://pypi.org/project/lexilensai-sdk/
- **Session Reconstruction Library:** https://github.com/curvsort/agent-session-graph
- **Documentation:** https://github.com/curvsort/lexilensai-sdk#readme

---

*Built by CurvSort | Apache 2.0 License*